# Medical LLM Assignment — 1A + 2A

| S. No | Name | Bits ID |
|-------|------|---------|
| 1 | Madhu Gururajan | 2025cs05027 |
| 2 | Anand Bharti | 2025cs05036 |

One notebook, **Runtime → Run all**. Assignment **1A** trains the domain model; Assignment **2A** prompts it and runs DPO. There are no `RUN_*` / `BACKEND` switches. Headings are prefixed **1A ·** and **2A ·** so the two briefs stay distinct.

| Assignment | What you run | Marks |
|---|---|---|
| **1A** | CPT on a ≥10 MB medical corpus, then three QLoRA adapters | 15 |
| **2A** | Five prompting strategies, structured reasoning, DPO on adapter B | 20 |
| **Total** | | **35** |

**Domain:** Medical & Clinical Literature — `microsoft/biogpt-large`.

**Hardware.** One 1.5B model in memory at a time. 1A unloads each QLoRA adapter before the next; 2A unloads SFT generations before DPO. CUDA (T4): ~60 CPT steps, 4-bit QLoRA, 5-sample self-consistency, 5 APE variants, 3 DPO epochs. **Apple MPS:** LoRA CPT, then the same adapter path with 3-sample self-consistency, 3 APE variants, ~20 DPO steps. CPU still trains 1A (short LoRA CPT/SFT); 2A Parts A/B stay offline on CPU, with 8 DPO steps if adapter B exists.


In [ ]:
# Colab's "Open notebook from GitHub" copies only this .ipynb, not src/.
# Clone the repo into /content so later cells can import src/.
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/madhu-GG/bits_llm_assignment_1_sem3.git"
REPO_NAME = "bits_llm_assignment_1_sem3"

def _has_src(root: Path) -> bool:
    return (root / "src" / "runtime.py").exists() and (root / "src" / "prompt_strategies.py").exists()

if _has_src(Path.cwd()):
    print("Already in the assignment folder:", Path.cwd())
elif "google.colab" in sys.modules:
    dest = Path("/content") / REPO_NAME
    if not _has_src(dest):
        print("Cloning", REPO_URL, "into", dest)
        env = os.environ.copy()
        env["GIT_TERMINAL_PROMPT"] = "0"
        clone_url = REPO_URL
        token = None
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")  # optional Colab secret; never hardcode a token
        except Exception:
            token = None
        if token:
            clone_url = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
        subprocess.run(["git", "clone", "--depth", "1", clone_url, str(dest)], check=True, env=env)
        if token:
            subprocess.run(["git", "-C", str(dest), "remote", "set-url", "origin", REPO_URL], check=True)
    os.chdir(dest)
    print("Working directory:", Path.cwd())
else:
    raise FileNotFoundError(
        "src/ is missing. Open this notebook from the assignment folder, or clone the GitHub repo first."
    )

assert _has_src(Path.cwd()), f"src/ still missing under {Path.cwd()}"


In [ ]:
# Install once on Colab. Locally, use: pip install -r requirements.txt
import sys
if "google.colab" in sys.modules:
    %pip -q install pypdf pyarrow matplotlib transformers accelerate peft trl datasets bitsandbytes sentencepiece sacremoses psutil huggingface_hub pandas


In [ ]:
# Memory env vars must be set BEFORE torch is imported.
from __future__ import annotations

import gc
import json
import hashlib
import logging
import math
import os
import random
import re
import sys
import warnings
import importlib
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

ROOT = Path.cwd()
print(f"We are running in {ROOT}")

# Refresh import caches after the Colab clone.
sys.path_importer_cache.clear()
importlib.invalidate_caches()

# Import `src.runtime` from the repo root, not from a stray src/ on sys.path.
abs_root = str(ROOT.resolve())
if abs_root not in sys.path:
    sys.path.insert(0, abs_root)

for path_str in list(sys.path):
    if path_str.endswith("/src") or path_str.endswith("\\src"):
        sys.path.remove(path_str)

from src.runtime import (
    apply_memory_env,
    cpt_steps_for_dataset,
    detect_runtime,
    disable_optional_scipy_backend,
    load_causal_lm,
    memory_snapshot,
    merge_lora_if_needed,
    model_audit,
    project_root,
    release_model_memory,
    resolve_lora_targets,
    seed_everything,
    trainer_memory_kwargs,
    training_budget,
    wrap_lora_for_cpt,
)
from src.corpus import MIN_CLEAN_BYTES, ensure_pdf_corpus, iter_source_pdfs

apply_memory_env()
seed_everything(42)

ROOT = project_root()
abs_root = str(ROOT.resolve())
if abs_root not in sys.path:
    sys.path.insert(0, abs_root)

RAW_PDFS = ROOT / "raw_pdfs"
EXTRACTED = ROOT / "extracted_text"
CLEAN_CORPUS = ROOT / "clean_corpus"
OUTPUTS = ROOT / "outputs"
for folder in (EXTRACTED, CLEAN_CORPUS, OUTPUTS):
    folder.mkdir(parents=True, exist_ok=True)

MODEL_ID = "microsoft/biogpt-large"
# Pin the checkpoint revision that ships safetensors.
MODEL_REVISION = "4585354702bf941a07727e316182421fea0051fe"
CPT_DIR = OUTPUTS / "biogpt-large-cpt"
PACKED_PARQUET = OUTPUTS / "medical_packed_train.parquet"
ADAPTER_ROOT = OUTPUTS / "qlora_adapters"

DOMAIN_PROMPTS = [
    "Explain why repeated blood-pressure measurements are useful in hypertension.",
    "What is the relationship between insulin resistance and type 2 diabetes?",
    "Why is antimicrobial stewardship important?",
]
GENERAL_PROMPTS = [
    "The capital of France is",
    "Water boils at",
    "The speed of light is approximately",
]
EXPECTED_TERMS = [["Paris"], ["100", "one hundred"], ["299", "300,000", "three hundred"]]

RUNTIME = detect_runtime()
BUDGET = training_budget(RUNTIME)
# 1024 on CUDA; 256 (or 128) on CPU so activations fit next to 6 GB fp32 weights.
SEQUENCE_LENGTH = int(BUDGET.get("sequence_length") or 1024)
src_path = str((ROOT / "src").resolve())
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(1, src_path)

print({"root": str(ROOT), "model": MODEL_ID, "runtime": RUNTIME, "budget": BUDGET, "sequence_length": SEQUENCE_LENGTH})
memory_snapshot("startup")


# Assignment 1A — CPT + QLoRA SFT  [15 marks]

**1A · Part A (10 marks)** — PDF corpus → clean ≥10 MB → pack tokens → inspect BioGPT-Large → continual pre-training (CPT) → domain perplexity and a forgetting check.

**1A · Part B (5 marks)** — ≥100 instruction pairs → three QLoRA adapters (A/B/C) → compare them to the Part A baseline.

The CPT checkpoint is written to `outputs/biogpt-large-cpt/`. Adapter B (`outputs/qlora_adapters/adapter_B`) is the policy Assignment 2A loads.


## 1A · Part A, Step 1 — Collect, extract, and clean PDFs (2 marks)

PDFs come from the public Hugging Face dataset `madhugg93/MedicalText` (`raw_pdfs.tgz`). The 12 `medical_topic_*.pdf` files are one-page starter notes (~0.04 MB) and are ignored. Cleaning order is the one the brief asks for: length → within-document repetition → exact-document dedup → English.


In [ ]:
class PdfWarningCollector(logging.Handler):
    def __init__(self):
        super().__init__(level=logging.WARNING)
        self.messages = {}

    def emit(self, record):
        message = record.getMessage()
        normalized = re.sub(r" at byte 0x[0-9a-f]+", " at byte <offset>", message)
        self.messages[normalized] = self.messages.get(normalized, 0) + 1


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def extract_pdfs_page_by_page(pdf_dir, extracted_dir):
    from pypdf import PdfReader
    collector = PdfWarningCollector()
    logger = logging.getLogger("pypdf")
    logger.addHandler(collector)
    logger.setLevel(logging.WARNING)
    logger.propagate = False
    extracted_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    pdfs = iter_source_pdfs(pdf_dir)
    for i, pdf_path in enumerate(pdfs, start=1):
        text_path = extracted_dir / f"{pdf_path.stem}.txt"

        # Skip extraction if the text file already exists to handle interrupted runs
        if text_path.exists():
            try:
                existing_text = text_path.read_text(encoding="utf-8", errors="ignore")
                # Simple heuristic to count pages using our markers if present
                page_count = max(1, len(re.findall(r"\[PAGE \d+\]", existing_text)))
                rows.append({"file": pdf_path.name, "status": "success", "pages": page_count, "characters": len(existing_text)})
                if i % 20 == 0 or i == len(pdfs):
                    print(f"  skipped {i}/{len(pdfs)} PDFs (already extracted)")
                continue
            except Exception:
                pass

        reader = None
        try:
            reader = PdfReader(str(pdf_path), strict=False)
            pages = []
            for n, page in enumerate(reader.pages, start=1):
                text = (page.extract_text() or "").replace("\x00", "").strip()
                pages.append(f"[PAGE {n}]\n{text}")
            extracted = "\n\n".join(pages)
            text_path.write_text(extracted, encoding="utf-8")
            rows.append({"file": pdf_path.name, "status": "success", "pages": len(pages), "characters": len(extracted)})
        except Exception as exc:
            text_path.unlink(missing_ok=True)
            rows.append({"file": pdf_path.name, "status": "error", "pages": 0, "characters": 0, "error": f"{type(exc).__name__}: {exc}"})
        reader = None
        if i % 20 == 0 or i == len(pdfs):
            print(f"  extracted {i}/{len(pdfs)} PDFs")
        if i % 25 == 0:
            gc.collect()
    return rows


def paragraphs(text):
    blocks = re.split(r"\n\s*\n+", text)
    return [re.sub(r"\s+", " ", block).strip() for block in blocks if block.strip()]


def looks_english(text):
    words = re.findall(r"[A-Za-z]+", text.lower())
    stopwords = {"the", "and", "of", "to", "in", "is", "for", "with", "on", "as", "by", "this", "an", "or", "from", "are", "be", "can", "which", "when"}
    if not words:
        return False
    ascii_ratio = sum(ord(ch) < 128 for ch in text) / max(len(text), 1)
    stopword_hits = sum(word in stopwords for word in words)
    return ascii_ratio >= 0.85 and stopword_hits >= max(2, min(10, len(words) // 30))


def clean_extracted_text(extracted_dir, cleaned_dir, min_chars=50, duplicate_fraction=0.30):
    files = sorted(p for p in extracted_dir.glob("*.txt") if not p.name.startswith("medical_topic_"))
    counts = {"before": len(files), "after_length": 0, "after_repetition": 0, "after_deduplication": 0, "after_language": 0}
    candidates = []
    for path in files:
        text = path.read_text(encoding="utf-8", errors="ignore").strip()
        if len(text) >= min_chars:
            candidates.append((path, text))
    counts["after_length"] = len(candidates)
    non_repetitive = []
    for path, text in candidates:
        paras = paragraphs(text)
        duplicate_ratio = (len(paras) - len(set(paras))) / max(len(paras), 1)
        if duplicate_ratio <= duplicate_fraction:
            non_repetitive.append((path, text))
    counts["after_repetition"] = len(non_repetitive)
    seen, unique = set(), []
    for path, text in non_repetitive:
        key = re.sub(r"\s+", " ", text).strip()
        if key not in seen:
            seen.add(key)
            unique.append((path, text))
    counts["after_deduplication"] = len(unique)
    final = [(path, text) for path, text in unique if looks_english(text)]
    counts["after_language"] = len(final)
    cleaned_dir.mkdir(parents=True, exist_ok=True)
    for old in cleaned_dir.glob("*.txt"):
        old.unlink()
    for path, text in final:
        (cleaned_dir / path.name).write_text(text, encoding="utf-8")
    impact = {
        "length_filter": counts["before"] - counts["after_length"],
        "repetition_filter": counts["after_length" ] - counts["after_repetition"],
        "deduplication": counts["after_repetition"] - counts["after_deduplication"],
        "language_filter": counts["after_deduplication"] - counts["after_language"],
    }
    greatest = max(impact, key=impact.get) if max(impact.values(), default=0) > 0 else "none (no documents removed)"
    return counts, impact, greatest


print("Collecting PDFs (local tarball, else Hugging Face madhugg93/MedicalText)...")
corpus_report = ensure_pdf_corpus(ROOT, RAW_PDFS)
source_pdfs = iter_source_pdfs(RAW_PDFS)
if len(source_pdfs) < 100:
    raise FileNotFoundError(
        f"Only {len(source_pdfs)} real PMC papers in {RAW_PDFS}. "
        "Need ~140 from madhugg93/MedicalText, not the 12 starter notes."
    )

extraction_stats = extract_pdfs_page_by_page(RAW_PDFS, EXTRACTED)
clean_counts, clean_impact, greatest_impact = clean_extracted_text(EXTRACTED, CLEAN_CORPUS)
corpus_bytes = sum(p.stat().st_size for p in CLEAN_CORPUS.glob("*.txt"))
step1 = {
    "pdf_documents": len(extraction_stats),
    "pages_extracted": sum(row["pages"] for row in extraction_stats),
    "pdfs_failed": sum(1 for row in extraction_stats if row["status"] == "error"),
    "clean_corpus_bytes": corpus_bytes,
    "clean_corpus_megabytes": round(corpus_bytes / 1024**2, 4),
    "cleaning_counts": clean_counts,
    "removed_by_step": clean_impact,
    "greatest_impact": greatest_impact,
}
if corpus_bytes < MIN_CLEAN_BYTES:
    warnings.warn("Clean corpus is still below 10 MB. CPT will not move perplexity on starter notes.")
print(json.dumps(step1, indent=2))
(OUTPUTS / "step1_cleaning_report.json").write_text(json.dumps(step1, indent=2), encoding="utf-8")
memory_snapshot("after_clean")


## 1A · Held-out split

10% of **documents** are held out for perplexity. The split is by file, not by packed chunk, so evaluation text is never seen during CPT.


In [ ]:
def split_corpus_files(cleaned_dir, eval_fraction=0.10, seed=42):
    files = sorted(cleaned_dir.glob("*.txt"))
    rng = random.Random(seed)
    order = files.copy()
    rng.shuffle(order)
    n_eval = max(1, int(round(len(order) * eval_fraction)))
    eval_files = sorted(order[:n_eval])
    train_files = sorted(order[n_eval:])
    manifest = {
        "train_documents": len(train_files),
        "eval_documents": len(eval_files),
        "train_files": [p.name for p in train_files],
        "eval_files": [p.name for p in eval_files],
    }
    (OUTPUTS / "corpus_split.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print({"train_documents": manifest["train_documents"], "eval_documents": manifest["eval_documents"]})
    return train_files, eval_files

train_files, eval_files = split_corpus_files(CLEAN_CORPUS)
eval_texts = [p.read_text(encoding="utf-8") for p in eval_files]


## 1A · Part A, Step 2 — Tokenize and pack (2 marks)

We reuse BioGPT's tokenizer (no custom vocab). Each document is wrapped with BOS/EOS, then all token ids are concatenated and sliced into packed blocks (`SEQUENCE_LENGTH` from the runtime budget: 1024 on CUDA, 256 on a typical CPU) so training is never padded with zeros.


In [ ]:
def encode_without_model_limit(tokenizer, text, chunk_chars=4000):
    ids = []
    for start in range(0, len(text), chunk_chars):
        ids.extend(tokenizer.encode(text[start:start + chunk_chars], add_special_tokens=False))
    return ids


def tokenize_and_pack(files, parquet_path, model_id, sequence_length):
    from transformers import AutoTokenizer
    import pyarrow as pa
    import pyarrow.parquet as pq

    disable_optional_scipy_backend()
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=MODEL_REVISION)
    bos_id, eos_id = tokenizer.bos_token_id, tokenizer.eos_token_id
    if bos_id is None:
        bos_id = eos_id if eos_id is not None else tokenizer.unk_token_id
        warnings.warn("Tokenizer has no BOS token; using EOS/UNK.")
    if eos_id is None:
        eos_id = bos_id
        warnings.warn("Tokenizer has no EOS token; using BOS.")

    stream, document_lengths, packed_ids, packed_mask = [], [], [], []
    for path in files:
        ids = [bos_id] + encode_without_model_limit(tokenizer, path.read_text(encoding="utf-8")) + [eos_id]
        document_lengths.append(len(ids))
        stream.extend(ids)
        while len(stream) >= sequence_length:
            chunk = stream[:sequence_length]
            packed_ids.append(chunk)
            packed_mask.append([1] * sequence_length)
            stream = stream[sequence_length:]
    remainder = len(stream)
    parquet_path.parent.mkdir(parents=True, exist_ok=True)
    pq.write_table(
        pa.table({"input_ids": packed_ids, "attention_mask": packed_mask}),
        parquet_path,
    )
    stats = {
        "tokenizer": model_id,
        "documents_packed": len(files),
        "total_tokens_in_stream": sum(document_lengths),
        "average_document_length_tokens": round(sum(document_lengths) / max(len(document_lengths), 1), 2),
        "packed_sequences": len(packed_ids),
        "sequence_length": sequence_length,
        "unused_remainder_tokens": remainder,
        "path": str(parquet_path),
    }
    del packed_ids, packed_mask, stream
    gc.collect()
    return tokenizer, stats

# Load the held-out split from disk if this cell is run on its own.
if "train_files" not in globals():
    train_files, eval_files = split_corpus_files(CLEAN_CORPUS)
    eval_texts = [p.read_text(encoding="utf-8") for p in eval_files]

tokenizer, packing_stats = tokenize_and_pack(train_files, PACKED_PARQUET, MODEL_ID, SEQUENCE_LENGTH)
print(json.dumps(packing_stats, indent=2))
(OUTPUTS / "packing_stats.json").write_text(json.dumps(packing_stats, indent=2), encoding="utf-8")
memory_snapshot("after_pack")


## 1A · Part A, Step 3 — Load the model, inspect it, save a baseline (2 marks)

On CUDA, load BioGPT-Large **directly onto the GPU** so the 1.5B weights never sit in host RAM. On CPU/MPS the same helper loads into host (or unified) memory. After the three domain prompts are saved, that copy is freed and CPT loads its own.


In [ ]:
def generate_outputs(model, tokenizer, prompts, max_new_tokens=60):
    import torch
    model.eval()
    if hasattr(model, "config"):
        model.config.use_cache = True
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    device = next(model.parameters()).device
    results = []
    for prompt in prompts:
        inputs = {k: v.to(device) for k, v in tokenizer(prompt, return_tensors="pt").items()}
        with torch.no_grad():
            generated = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        results.append({"prompt": prompt, "generated_text": tokenizer.decode(generated[0], skip_special_tokens=True)})
    return results


from transformers import AutoTokenizer

disable_optional_scipy_backend()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

if not BUDGET.get("can_load_model", True):
    print("Skipping model load:", BUDGET["reason"])
else:
    base_model = load_causal_lm(MODEL_ID, MODEL_REVISION, four_bit=False, gradient_checkpointing=False, runtime=RUNTIME)
    audit = {"model_id": MODEL_ID, "device": RUNTIME["device"], "dtype": RUNTIME["dtype"], **model_audit(base_model)}
    print(json.dumps(audit, indent=2))
    (OUTPUTS / "model_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")

    baseline = generate_outputs(base_model, tokenizer, DOMAIN_PROMPTS)
    (OUTPUTS / "baseline_outputs.json").write_text(json.dumps(baseline, indent=2), encoding="utf-8")
    for row in baseline:
        print("---")
        print(row["prompt"])
        print(row["generated_text"][:400])

    release_model_memory(base_model)
    base_model = None
memory_snapshot("after_baseline")


## 1A · Part A, Step 4 — Continual pre-training (2 marks)

Hugging Face `Trainer` + AdamW (8-bit on T4) + linear warmup. A custom callback records loss. Starting loss should be ~2–4; ~10.8 means the weights were random.

**How many steps?** `src/runtime.py` picks this from the machine:
- Colab T4: **60 steps** (~1 hour at the measured ~57 s/step), 8-bit Adam, full-weight CPT.
- A100/L40S: one packed epoch (~350 steps), full AdamW.
- **CPU / Apple MPS:** full AdamW on 1.5B fp32 does not fit (~6 GB weights + ~6 GB grads + ~12 GB Adam). CPT therefore uses **LoRA** (`q_proj`/`v_proj`, r=8), sequence length **256**, batch 1, **8 steps** (16 on MPS), then merges the adapter back into the base weights so the saved `biogpt-large-cpt/` folder is a normal model. Slow, but Run all completes without a GPU.


In [ ]:
import torch
import gc
import sys
from transformers import AutoTokenizer

# Free the baseline copy before CPT so only one 1.5B model is resident.
for var_name in ["cpt_model", "base_model", "trainer", "model"]:
    if var_name in globals():
        print(f"Dropping {var_name} before CPT")
        del globals()[var_name]

# Keep (or reload) the tokenizer; it is small and CPT reuses it.
if "tokenizer" not in globals():
    print("Loading tokenizer.")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
else:
    print("Tokenizer kept.")

# Drop IPython last-result refs that can keep the old model alive.
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.user_ns.pop('_', None)
        ipython.user_ns.pop('__', None)
        ipython.user_ns.pop('___', None)
        # Clear input/output history cells too.
        ipython.user_ns['_ih'] = []
        ipython.user_ns['_oh'] = {}
        ipython.user_ns['_dh'] = []
        print("IPython last-result refs dropped.")
except Exception as e:
    print(f"Could not clear IPython cache: {e}")

# Collect unreachable tensors.
for _ in range(5):
    gc.collect()

# Return unused CUDA blocks to the allocator.
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print("Ready for CPT; tokenizer kept.")
from src.runtime import memory_snapshot
memory_snapshot("after_explicit_cleanup")


In [ ]:
class PackedTextDataset:
    # Memory-map the packed Parquet so the trainer streams sequences from disk.

    def __new__(cls, parquet_path):
        import torch
        import pyarrow.parquet as pq

        table = pq.read_table(parquet_path, memory_map=True)
        ids_col = table.column("input_ids")
        mask_col = table.column("attention_mask")

        class Dataset(torch.utils.data.Dataset):
            def __len__(self):
                return len(ids_col)

            def __getitem__(self, index):
                ids = torch.tensor(ids_col[index].as_py(), dtype=torch.long)
                mask = torch.tensor(mask_col[index].as_py(), dtype=torch.long)
                return {"input_ids": ids, "attention_mask": mask, "labels": ids.clone()}

        return Dataset()


def train_cpt(model, tokenizer, dataset, output_dir, max_steps, learning_rate=5e-5):
    from transformers import Trainer, TrainerCallback, TrainingArguments

    class LossHistoryCallback(TrainerCallback):
        def __init__(self):
            self.history = []

        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs and "loss" in logs:
                self.history.append({"step": int(state.global_step), "loss": float(logs["loss"])})

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    callback = LossHistoryCallback()
    mem_kwargs = trainer_memory_kwargs(RUNTIME, BUDGET)

    # 8-bit Adam is required on a T4 so optimiser states fit. Faster GPUs use
    # full AdamW. CPU/MPS keep adamw_torch on the LoRA params only.
    opt_val = mem_kwargs.get("optim", "adamw_torch")
    if BUDGET.get("eight_bit_adam"):
        try:
            import bitsandbytes  # noqa: F401
            opt_val = "adamw_bnb_8bit"
        except ImportError:
            warnings.warn("bitsandbytes not found; using adamw_torch. CPT may OOM on a 15 GB T4.")
            opt_val = "adamw_torch"

    # Warmup length as a step count: budget ratio × max_steps.
    ratio = BUDGET.get("warmup_ratio", 0.1)
    computed_warmup_steps = int(max_steps * ratio)

    args_kwargs = dict(
        output_dir=str(output_dir),
        max_steps=max_steps,
        learning_rate=learning_rate,
        per_device_train_batch_size=BUDGET.get("per_device_train_batch_size", 1),
        gradient_accumulation_steps=BUDGET.get("gradient_accumulation_steps", 8),
        warmup_steps=computed_warmup_steps,
        weight_decay=mem_kwargs.get("weight_decay", 0.01),
        optim=opt_val,
        fp16=mem_kwargs.get("fp16", False),
        bf16=mem_kwargs.get("bf16", False),
        gradient_checkpointing=True,
        logging_steps=mem_kwargs.get("logging_steps", 10),
        save_strategy=mem_kwargs.get("save_strategy", "no"),
        report_to=mem_kwargs.get("report_to", "none"),
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
        remove_unused_columns=False,
    )
    if mem_kwargs.get("use_cpu"):
        args_kwargs["use_cpu"] = True
    try:
        args = TrainingArguments(**args_kwargs)
    except TypeError:
        args_kwargs.pop("use_cpu", None)
        args = TrainingArguments(**args_kwargs)

    trainer = Trainer(model=model, args=args, train_dataset=dataset, callbacks=[callback])
    trainer.train()
    # CPU LoRA-CPT is merged back so later cells load a normal causal LM folder.
    trainer.model = merge_lora_if_needed(trainer.model)
    output_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    history = callback.history
    del trainer
    return history


def plot_loss_history(history, output_path):
    import matplotlib.pyplot as plt
    if not history:
        warnings.warn("No loss records; skip plot.")
        return
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot([row["step"] for row in history], [row["loss"] for row in history], marker="o", markersize=3)
    ax.set_title("CPT training loss")
    ax.set_xlabel("Training step")
    ax.set_ylabel("Loss")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(output_path, dpi=140)
    plt.show()
    plt.close(fig)


# Load packing_stats from disk if this cell is run on its own.
if "packing_stats" not in globals():
    stats_path = OUTPUTS / "packing_stats.json"
    if stats_path.exists():
        packing_stats = json.loads(stats_path.read_text(encoding="utf-8"))
    else:
        raise NameError("packing_stats is not defined in memory, and packing_stats.json is missing.")

n_sequences = packing_stats["packed_sequences"]
cpt_steps = cpt_steps_for_dataset(n_sequences, BUDGET)
print({"packed_sequences": n_sequences, "cpt_steps": cpt_steps, "gpu_tier": BUDGET.get("gpu_tier"), "why": BUDGET["reason"]})

if cpt_steps <= 0:
    print("Not enough memory to train BioGPT-Large on this machine. Data prep is done; CPT/SFT need ~10 GB RAM or a GPU.")
else:
    cpt_model = load_causal_lm(MODEL_ID, MODEL_REVISION, four_bit=False, gradient_checkpointing=True, runtime=RUNTIME)
    cpt_model = wrap_lora_for_cpt(cpt_model, BUDGET)
    history = train_cpt(cpt_model, tokenizer, PackedTextDataset(PACKED_PARQUET), CPT_DIR, max_steps=cpt_steps)
    (OUTPUTS / "cpt_loss_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
    plot_loss_history(history, OUTPUTS / "cpt_loss_curve.png")
    print({
        "initial_logged_loss": history[0]["loss"] if history else None,
        "final_logged_loss": history[-1]["loss"] if history else None,
        "steps": cpt_steps,
        "checkpoint": str(CPT_DIR),
    })
    release_model_memory(cpt_model)
    cpt_model = None
memory_snapshot("after_cpt")


## 1A · Part A, Step 5 — Domain perplexity and catastrophic forgetting (2 marks)

Lower domain PPL after CPT means the model learned this literature. The same three general prompts check that it did not forget simple world knowledge. Base and CPT are loaded **one after the other**, never together.


In [ ]:
def perplexity(model, tokenizer, texts, max_length=1024):
    import torch
    model.eval()
    total_nll, total_tokens = 0.0, 0
    device = next(model.parameters()).device
    for text in texts:
        ids = encode_without_model_limit(tokenizer, text)
        for start in range(0, len(ids), max_length):
            chunk = ids[start:start + max_length]
            if len(chunk) < 2:
                continue
            input_ids = torch.tensor([chunk], dtype=torch.long, device=device)
            with torch.no_grad():
                logits = model(input_ids=input_ids).logits[:, :-1, :]
            targets = input_ids[:, 1:]
            loss_sum = torch.nn.functional.cross_entropy(
                logits.reshape(-1, logits.size(-1)), targets.reshape(-1), reduction="sum"
            )
            total_nll += float(loss_sum.item())
            total_tokens += targets.numel()
            del logits, input_ids, targets, loss_sum
    mean_nll = total_nll / max(total_tokens, 1)
    return {"mean_nll": mean_nll, "perplexity": math.exp(min(mean_nll, 20)), "tokens": total_tokens}


def forgetting_verdict(base_text, cpt_text, expected_terms):
    base_hits = sum(term.lower() in base_text.lower() for term in expected_terms)
    cpt_hits = sum(term.lower() in cpt_text.lower() for term in expected_terms)
    return "Degraded" if base_hits > 0 and cpt_hits == 0 else "Retained"


if not (CPT_DIR / "config.json").exists():
    print("CPT checkpoint not found; skipping evaluation. Run Step 4 first.")
else:
    base_model = load_causal_lm(MODEL_ID, MODEL_REVISION, four_bit=False, gradient_checkpointing=False, runtime=RUNTIME)
    base_ppl = perplexity(base_model, tokenizer, eval_texts, SEQUENCE_LENGTH)
    base_general = generate_outputs(base_model, tokenizer, GENERAL_PROMPTS, max_new_tokens=30)
    release_model_memory(base_model)
    base_model = None
    memory_snapshot("after_base_ppl")

    cpt_model = load_causal_lm(str(CPT_DIR), revision=None, four_bit=False, gradient_checkpointing=False, runtime=RUNTIME)
    cpt_ppl = perplexity(cpt_model, tokenizer, eval_texts, SEQUENCE_LENGTH)
    cpt_general = generate_outputs(cpt_model, tokenizer, GENERAL_PROMPTS, max_new_tokens=30)
    release_model_memory(cpt_model)
    cpt_model = None

    reduction = 100 * (base_ppl["perplexity"] - cpt_ppl["perplexity"]) / max(base_ppl["perplexity"], 1e-12)
    comparison = []
    for i, prompt in enumerate(GENERAL_PROMPTS):
        comparison.append({
            "prompt": prompt,
            "base_output": base_general[i]["generated_text"],
            "cpt_output": cpt_general[i]["generated_text"],
            "verdict": forgetting_verdict(base_general[i]["generated_text"], cpt_general[i]["generated_text"], EXPECTED_TERMS[i]),
        })
    results = {
        "base_perplexity": base_ppl,
        "cpt_perplexity": cpt_ppl,
        "ppl_reduction_percent": round(reduction, 2),
        "general_domain_comparison": comparison,
    }
    (OUTPUTS / "evaluation_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
    print(json.dumps(results, indent=2))
memory_snapshot("after_eval")


## 1A · Part B1 — Instruction dataset (2 marks)

Heuristic pairs taken only from the cleaned corpus (no extra LLM). At least 100 rows, 80/20 split, JSONL.


In [ ]:
def _normalize(text):
    return re.sub(r"\s+", " ", text).strip()


def _sentences(text):
    chunks = re.split(r"(?<=[.!?])\s+", _normalize(text))
    return [c.strip() for c in chunks if len(c.strip()) > 20]


def build_instruction_pairs(docs, max_pairs=120, min_chars=80):
    templates = [
        "Summarize this clinical point from domain text: {topic}",
        "Explain this medical statement in clear terms: {topic}",
        "What does the following domain evidence imply? {topic}",
        "Provide a concise evidence-based explanation: {topic}",
        "Based only on the corpus, clarify: {topic}",
    ]
    pairs, seen = [], set()
    for file_name, text in docs:
        for sentence in _sentences(text):
            if len(sentence) < min_chars:
                continue
            topic = sentence[:120].strip()
            instruction = templates[len(pairs) % len(templates)].format(topic=topic)
            response = sentence if len(sentence) <= 450 else sentence[:450].rsplit(" ", 1)[0] + "..."
            key = (instruction.lower(), response.lower())
            if key in seen:
                continue
            seen.add(key)
            pairs.append({"instruction": instruction, "response": response, "source_file": file_name})
            if len(pairs) >= max_pairs:
                return pairs
    return pairs


def write_jsonl(rows, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps({"instruction": row["instruction"], "response": row["response"]}, ensure_ascii=False) + "\n")


docs = [(p.name, p.read_text(encoding="utf-8", errors="ignore")) for p in sorted(CLEAN_CORPUS.glob("*.txt"))]
pairs = build_instruction_pairs(docs, max_pairs=120)
if len(pairs) < 100:
    raise ValueError(f"Only {len(pairs)} instruction pairs; need ≥100.")
rng = random.Random(42)
shuffled = pairs.copy()
rng.shuffle(shuffled)
split_at = max(1, min(len(shuffled) - 1, int(round(len(shuffled) * 0.8))))
train_rows, eval_rows = shuffled[:split_at], shuffled[split_at:]
write_jsonl(pairs, OUTPUTS / "instruction_dataset.jsonl")
write_jsonl(train_rows, OUTPUTS / "instruction_train.jsonl")
write_jsonl(eval_rows, OUTPUTS / "instruction_eval.jsonl")
b1 = {
    "documents_used": len(docs),
    "total_pairs": len(pairs),
    "train_pairs": len(train_rows),
    "eval_pairs": len(eval_rows),
    "method": "heuristic sentences from cleaned corpus",
    "synthetic_prompt_template": (
        "Read the text below and generate 10 instruction-response pairs in JSON format based ONLY on this text. "
        "Each entry must have instruction and response keys."
    ),
}
print(json.dumps(b1, indent=2))
(OUTPUTS / "dataset_counts.json").write_text(json.dumps(b1, indent=2), encoding="utf-8")


## 1A · Part B2 — Three QLoRA adapters (2 marks)

The brief asks for `q_proj` / `v_proj` / `o_proj`. BioGPT names the last of those **`out_proj`**, so adapter C trains `out_proj`. Each adapter is trained, evaluated on the three domain prompts, then **unloaded** before the next one starts. On CUDA, 4-bit NF4 keeps the 1.5B backbone inside 15 GB. On CPU/MPS there is no bitsandbytes, so the same LoRA adapters train on the fp32 backbone (only LoRA params have gradients).


In [ ]:
@dataclass(frozen=True)
class AdapterSpec:
    name: str
    rank: int
    alpha: int
    target_modules: tuple[str, ...]


ADAPTER_SPECS = [
    AdapterSpec("A", 8, 16, ("q_proj", "v_proj")),
    AdapterSpec("B", 16, 32, ("q_proj", "v_proj")),
    AdapterSpec("C", 32, 32, ("q_proj", "v_proj", "o_proj")),  # o_proj → out_proj on BioGPT
]


def _format_for_sft(tokenizer, instruction, response):
    messages = [{"role": "user", "content": instruction}, {"role": "assistant", "content": response}]
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        except Exception:
            pass
    return f"User: {instruction}\nAssistant: {response}"


def train_qlora_adapter(spec: AdapterSpec, train_rows, eval_rows, output_dir, max_steps, prompts):
    import inspect
    import torch
    from datasets import Dataset
    from peft import LoraConfig, get_peft_model
    from transformers import AutoTokenizer, TrainingArguments
    from trl import SFTTrainer
    try:
        from trl import SFTConfig
    except ImportError:
        SFTConfig = None

    sft_source = str(CPT_DIR) if (CPT_DIR / "config.json").exists() else MODEL_ID
    revision = None if sft_source == str(CPT_DIR) else MODEL_REVISION
    tok_src = sft_source if (Path(sft_source) / "tokenizer_config.json").exists() else MODEL_ID
    tok_kwargs = {} if Path(tok_src).exists() else {"revision": MODEL_REVISION}
    tokenizer = AutoTokenizer.from_pretrained(tok_src, **tok_kwargs)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    use_4bit = bool(BUDGET.get("four_bit")) and bool(RUNTIME.get("cuda"))
    model = load_causal_lm(sft_source, revision, four_bit=use_4bit, gradient_checkpointing=True, runtime=RUNTIME)
    if use_4bit:
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(model)
    targets = resolve_lora_targets(model, list(spec.target_modules))
    print(f"Adapter {spec.name}: LoRA r={spec.rank} α={spec.alpha} targets={targets}")
    model = get_peft_model(
        model,
        LoraConfig(r=spec.rank, lora_alpha=spec.alpha, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM", target_modules=targets),
    )

    train_ds = Dataset.from_list([{"text": _format_for_sft(tokenizer, r["instruction"], r["response"])} for r in train_rows])
    eval_ds = Dataset.from_list([{"text": _format_for_sft(tokenizer, r["instruction"], r["response"])} for r in eval_rows])

    tkwargs = trainer_memory_kwargs(RUNTIME, BUDGET)
    tkwargs.update({"max_steps": max_steps, "learning_rate": 2e-4, "output_dir": str(output_dir)})
    # SFT only trains LoRA. 8-bit/paged Adam is CUDA+bitsandbytes; CPU uses adamw_torch.
    if BUDGET.get("eight_bit_adam") and RUNTIME.get("cuda"):
        tkwargs["optim"] = "paged_adamw_8bit"
    else:
        tkwargs["optim"] = BUDGET.get("optim") or "adamw_torch"
    # Warmup length as a step count: budget ratio × max_steps.
    ratio = float(tkwargs.pop("warmup_ratio", BUDGET.get("warmup_ratio", 0.1)) or 0.1)
    tkwargs["warmup_steps"] = max(1, int(max_steps * ratio))
    args_cls = TrainingArguments
    if SFTConfig is not None:
        params = inspect.signature(SFTConfig).parameters
        if "dataset_text_field" in params or "max_length" in params:
            args_cls = SFTConfig
            if "dataset_text_field" in params:
                tkwargs["dataset_text_field"] = "text"
            if "max_length" in params:
                tkwargs["max_length"] = int(BUDGET.get("sft_max_length") or 512)
    # Instruction pairs are short, so SFT uses a smaller max length than CPT.
    while True:
        try:
            args = args_cls(**tkwargs)
            break
        except TypeError as err:
            msg = str(err)
            dropped = None
            if "unexpected keyword argument" in msg:
                dropped = msg.rsplit("'", 2)[-2] if msg.count("'") >= 2 else None
            if not dropped or dropped not in tkwargs:
                raise
            tkwargs.pop(dropped, None)
    trainer_kwargs = {"model": model, "args": args, "train_dataset": train_ds, "eval_dataset": eval_ds}
    tparams = inspect.signature(SFTTrainer.__init__).parameters
    if "processing_class" in tparams:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in tparams:
        trainer_kwargs["tokenizer"] = tokenizer
    if "dataset_text_field" in tparams:
        trainer_kwargs["dataset_text_field"] = "text"
    if "max_seq_length" in tparams:
        trainer_kwargs["max_seq_length"] = int(BUDGET.get("sft_max_length") or 512)

    trainer = SFTTrainer(**trainer_kwargs)
    trainer.train()
    output_dir.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    eval_out = {prompt: generate_outputs(model, tokenizer, [prompt], max_new_tokens=80)[0]["generated_text"] for prompt in prompts}
    release_model_memory(trainer, model, tokenizer, train_ds, eval_ds)
    return eval_out


sft_steps = BUDGET["sft_max_steps"]
eval_cache = {}
if sft_steps <= 0:
    print("Not enough memory to train LoRA adapters on this machine. Part B1 JSONL files are already written.")
else:
    ADAPTER_ROOT.mkdir(parents=True, exist_ok=True)
    trained = {}
    for spec in ADAPTER_SPECS:
        out_dir = ADAPTER_ROOT / f"adapter_{spec.name}"
        print(f"Training adapter {spec.name} for {sft_steps} steps...")
        eval_cache[spec.name] = train_qlora_adapter(spec, train_rows, eval_rows, out_dir, sft_steps, DOMAIN_PROMPTS)
        trained[spec.name] = str(out_dir)
        memory_snapshot(f"after_adapter_{spec.name}")
    (OUTPUTS / "trained_adapters.json").write_text(json.dumps(trained, indent=2), encoding="utf-8")
    (OUTPUTS / "b2_evaluation_cache.json").write_text(json.dumps(eval_cache, indent=2), encoding="utf-8")
    print("B2 complete:", trained)


## 1A · Part B3 — Compare adapters to the Part A baseline (1 mark)

Uses the generations cached in B2, so this cell does **not** load the 1.5B model again.


In [ ]:
import csv

baseline_path = OUTPUTS / "baseline_outputs.json"
cache_path = OUTPUTS / "b2_evaluation_cache.json"
if not baseline_path.exists():
    print("Baseline outputs missing; run Step 3.")
else:
    baseline_rows = json.loads(baseline_path.read_text(encoding="utf-8"))
    baseline_by_prompt = {row.get("prompt", ""): row.get("generated_text", "") for row in baseline_rows}
    evaluation_cache = json.loads(cache_path.read_text(encoding="utf-8")) if cache_path.exists() else eval_cache
    comparison = []
    for prompt in DOMAIN_PROMPTS:
        row = {"prompt": prompt, "baseline_output": baseline_by_prompt.get(prompt, "")}
        for spec in ADAPTER_SPECS:
            row[f"adapter_{spec.name}_output"] = evaluation_cache.get(spec.name, {}).get(prompt, "[adapter not trained — B2 was skipped]")
        comparison.append(row)
    with (OUTPUTS / "adapter_comparison.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(comparison[0]))
        writer.writeheader()
        writer.writerows(comparison)
    (OUTPUTS / "adapter_comparison.json").write_text(json.dumps(comparison, indent=2), encoding="utf-8")
    try:
        import pandas as pd
        display(pd.DataFrame(comparison))
    except Exception:
        print(json.dumps(comparison, indent=2)[:4000])
    print("Best adapter is a judgement call: pick the one whose answers stay closest to the corpus and the question.")
memory_snapshot("done")


## 1A · Deliverables written under `outputs/`

| File | Step |
|------|------|
| `step1_cleaning_report.json` | A1 |
| `medical_packed_train.parquet` | A2 |
| `model_audit.json`, `baseline_outputs.json` | A3 |
| `biogpt-large-cpt/`, `cpt_loss_curve.png` | A4 |
| `evaluation_results.json` | A5 |
| `instruction_dataset.jsonl` (+ train/eval) | B1 |
| `qlora_adapters/adapter_{A,B,C}/` | B2 |
| `adapter_comparison.csv` | B3 |

On a Colab T4, CPT is about **60 steps (~1 hour)** at the measured ~57 s/step, then three short QLoRA runs. On A100/L40S, CPT is about one packed epoch. If there is no GPU, everything up through the JSONL dataset still runs.


# Assignment 2A — Prompt Engineering & Preference Alignment  [20 marks]

| Part | What you run | Marks |
|---|---|---|
| A | Five prompting strategies × 5 queries, 2-step chain, function-calling JSON | 5 |
| B | CoT, ToT, Step-Back, PAL, ReACT, Self-Reflection | 8 |
| C | ≥60 preference pairs, DPO on adapter B, 5 held-out comparisons | 7 |

**How generation is chosen.** After 1A writes `outputs/qlora_adapters/adapter_B`, CUDA and Apple MPS use that adapter. Otherwise 2A uses a deterministic corpus-grounded **offline** backend so CPU Run all still finishes. Offline text is a reproducible stand-in; every saved artefact records which backend produced it.


In [ ]:
# Load adapter_B for prompting and DPO, or the offline backend if it is not on disk.
from a2_paths import DEFAULT_SFT_ADAPTER, DPO_ADAPTER_DIR, PREFERENCE_DATASET, base_model_source, sft_adapter_path
from domain_lm import DomainLM, choose_generation_backend, wrap

src_path = str((ROOT / "src").resolve())
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(1, src_path)

SFT_ADAPTER = DEFAULT_SFT_ADAPTER
ADAPTER_DIR = sft_adapter_path(SFT_ADAPTER)
ARTIFACTS = OUTPUTS
ARTIFACTS.mkdir(parents=True, exist_ok=True)

BACKEND = choose_generation_backend(ADAPTER_DIR, RUNTIME)
SC_SAMPLES = int(BUDGET.get("a2_self_consistency_samples") or 5)
APE_VARIANTS = int(BUDGET.get("a2_ape_variants") or 5)
REACT_CYCLES = int(BUDGET.get("a2_react_cycles") or 5)
A2_NEW_TOKENS = int(BUDGET.get("a2_max_new_tokens") or 200)
lm = DomainLM(
    base_model_id=base_model_source(),
    adapter_path=ADAPTER_DIR if BACKEND == "adapter" else None,
    backend=BACKEND,
    label=f"sft_{SFT_ADAPTER}",
)
print(json.dumps({"runtime": RUNTIME, "budget_tier": BUDGET.get("gpu_tier"),
                  "adapter": str(ADAPTER_DIR), "adapter_exists": ADAPTER_DIR.exists(),
                  "sc_samples": SC_SAMPLES, "ape_variants": APE_VARIANTS,
                  "dpo_max_steps": BUDGET.get("dpo_max_steps"),
                  "lm": lm.describe()}, indent=2, default=str))
if BACKEND == "offline" and (RUNTIME.get("cuda") or RUNTIME.get("mps")) and not ADAPTER_DIR.exists():
    print("1A did not write adapter_B. Assignment 2A Parts A/B use the offline backend.")
memory_snapshot("a2_startup")


## 2A · Part A — Prompt paradigms & system design  [5 marks]

Load the Assignment 1 adapter (or the offline stand-in) and apply **five prompting strategies** to the same five domain queries. Decoding is greedy (`temperature=0`) except self-consistency at `0.7`. Token length and sample counts come from the runtime budget: CUDA keeps 200 tokens / 5 SC samples / 5 APE variants; MPS uses 128 / 3 / 3 so a local run stays near an hour.

| Strategy | Configuration | Expected effect |
|---|---|---|
| Zero-shot | No examples; task described in the instruction only | Tests raw adapter capability |
| Few-shot (3-shot) | 3 domain Q&A examples prepended | Improves format adherence |
| Instruction prompting | Explicit role + step-by-step instruction format | Improves structure |
| Self-consistency | Same prompt N× at `temperature=0.7`, majority vote (N=5 CUDA, N=3 MPS) | Reduces variance |
| APE | N LLM-generated prompt variants, best selected by output quality (N=5 CUDA, N=3 MPS) | Optimises instruction wording |

Answers are scored by one shared rubric (`src/answer_scoring.py`) out of 100: grounding 35, structure 20, completeness 20, specificity 15, non-repetition 10. The same scorer is reused for ToT / APE selection and for the chosen/rejected decision in Part C.


In [ ]:
import pandas as pd
import prompt_strategies as ps
from assignment2_queries import DOMAIN_QUERIES

# Decode length follows the runtime budget (200 tokens on CUDA, shorter on MPS/CPU).
ps.MAX_NEW_TOKENS = A2_NEW_TOKENS

def show(frame):
    try:
        display(frame)
    except Exception:
        print(frame.to_string(index=False))

for i, q in enumerate(DOMAIN_QUERIES, 1):
    print(f"Q{i}. {q}")


In [ ]:
# 5 queries × 5 strategies. CUDA: 5 SC samples / 5 APE variants. MPS: 3 / 3.
a1_rows = ps.run_all_strategies(lm, DOMAIN_QUERIES, sc_samples=SC_SAMPLES, ape_variants=APE_VARIANTS)
print(f"{len(a1_rows)} strategy runs = {len(DOMAIN_QUERIES)} queries × {len(ps.STRATEGIES)} strategies "
      f"(self-consistency n={SC_SAMPLES}, APE variants={APE_VARIANTS})")


In [ ]:
table = pd.DataFrame([{
    "query": f"Q{DOMAIN_QUERIES.index(r['query']) + 1}",
    "strategy": r["strategy"],
    "score": r["score"]["total"],
    "grounding": r["score"]["grounding"],
    "structure": r["score"]["structure"],
    "words": r["score"]["words"],
    "answer": " ".join(r["answer"].split())[:110] + "...",
} for r in a1_rows])
show(table.pivot(index="query", columns="strategy", values="score"))
show(pd.DataFrame(ps.strategy_summary(a1_rows)))


In [ ]:
# Full text for one query so the qualitative difference is visible, not just the score.
focus = DOMAIN_QUERIES[1]
for r in [r for r in a1_rows if r["query"] == focus]:
    print("=" * 100)
    print(r["strategy"], "| score", r["score"]["total"])
    print(wrap(r["answer"]))


In [ ]:
# Self-consistency detail: the samples, the vote, and the agreement rate.
sc = ps.run_self_consistency(lm, focus, samples=SC_SAMPLES)
print("agreement:", sc["agreement"], "| majority cluster size:", sc["majority_size"], f"/ {SC_SAMPLES}")
print("vote counts:", {str(k)[:40]: v for k, v in sc["vote_counts"].items()})
for i, sample in enumerate(sc["samples"], 1):
    print(f"\n--- sample {i} ---\n", wrap(sample[:400]))


In [ ]:
# APE detail: the generated prompt variants and the score of each resulting answer.
ape = ps.run_ape(lm, focus, num_variants=APE_VARIANTS)
print("variants tried:", ape["variants_tried"], "| scores:", ape["variant_scores"])
print("\nRaw LLM proposals:\n", wrap(ape["raw_proposals"]))
print("\nSelected prompt:\n", wrap(ape["selected_prompt"]))


### 2A · A1 write-up

Read the pivot table and the summary, then record:

1. **Zero-shot** is the floor: short, unstructured, no limiting condition, no caveat.
2. **Few-shot** mainly buys format adherence; grounding moves less than structure.
3. **Instruction prompting** is the cheapest structure win (one generation).
4. **Self-consistency** removes unlucky samples. Report the agreement rate — a high rate means the extra 5× cost is not justified.
5. **APE** wins when the hand-written instruction was not optimal; look at the spread of `variant_scores`.


## 2A · A2 — Prompt chaining and function calling  [2 marks]

**Chain:** Step 1 extracts facts from retrieved corpus context → Step 2 synthesises the answer *only* from those facts. The same complex query is also run as a single-step prompt so the gain is from decomposition, not from extra context.

**Function call:** the model must return JSON matching `{entity, value, unit, source_clause, confidence}`. Invalid JSON is fed back as a repair turn.


In [ ]:
from assignment2_queries import CHAIN_QUERY, CLINICAL_FACT_SCHEMA

chain = ps.run_prompt_chain(lm, CHAIN_QUERY)
print("COMPLEX QUERY:\n", wrap(CHAIN_QUERY))
print("\n--- STEP 1: extracted facts ---\n", wrap(chain["step1_facts"]))
print("\n--- STEP 2: synthesised answer ---\n", wrap(chain["chained_answer"]))
print("\n--- SINGLE-STEP BASELINE ---\n", wrap(chain["single_step_answer"]))


In [ ]:
comparison = pd.DataFrame([
    {"approach": "single-step prompt", **chain["single_step_score"]},
    {"approach": "2-step chain", **chain["chained_score"]},
])
show(comparison)
delta = chain["chained_score"]["total"] - chain["single_step_score"]["total"]
print(f"Chaining changes the rubric score by {delta:+.2f} points.")


In [ ]:
print(json.dumps(CLINICAL_FACT_SCHEMA, indent=2))
call = ps.run_function_call(lm, "Extract one clinical fact about diagnostic test performance.")
print("valid:", call["valid"], "| attempts:", len(call["attempts"]))
for attempt in call["attempts"]:
    print(f"\nattempt {attempt['attempt']} valid={attempt['valid']}"
          + (f" error={attempt.get('error')}" if not attempt["valid"] else ""))
    print(wrap(attempt["raw"][:400]))
print("\nParsed object:\n", json.dumps(call["parsed"], indent=2))


In [ ]:
# Negative control: the validator must reject malformed or off-schema output.
for bad in ['{"entity": "x"}', "not json at all",
            '{"entity":"x","value":"y","unit":"z","source_clause":"c","confidence":"high"}']:
    try:
        ps.parse_structured_output(bad)
        print("ACCEPTED (unexpected):", bad)
    except ps.SchemaValidationError as exc:
        print("rejected:", exc)


In [ ]:
artefact_a = {
    "backend": lm.describe(),
    "a1_rows": [{k: v for k, v in r.items() if k != "samples"} for r in a1_rows],
    "a1_summary": ps.strategy_summary(a1_rows),
    "a2_chain": chain,
    "a2_function_call": call,
}
path = ARTIFACTS / "part_a_results.json"
path.write_text(json.dumps(artefact_a, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("saved", path)
memory_snapshot("after_part_a")


## 2A · Part B — Structured & iterative reasoning  [8 marks]

Same five Part A queries plus two multi-step problems that need decomposition.

| Step | Deliverable | Marks |
|---|---|---|
| B1 | CoT, ToT, Step-Back × 2 domain problems + comparison table | 4 |
| B2 | PAL (executed Python) + ReACT (≥3 cycles) + Self-Reflection | 4 |


In [ ]:
import reasoning_strategies as rs
from assignment2_queries import PAL_PROBLEMS, REACT_QUESTION, REASONING_PROBLEMS

rs.MAX_NEW_TOKENS = A2_NEW_TOKENS

for p in REASONING_PROBLEMS:
    print(p["id"], "->", wrap(p["question"]), "\n")


## 2A · B1 — Chain-of-Thought

`Let's think step by step.` is appended; the model numbers each step and finishes with `Answer:`. The same query is also run with a plain prompt.

**When to use it:** any multi-step clinical query where the conclusion depends on chaining a definition to a limiting condition (dose selection, test interpretation, risk weighing).


In [ ]:
cot_rows = [rs.run_cot(lm, p["question"]) for p in REASONING_PROBLEMS]
cot_rows += [rs.run_cot(lm, q) for q in DOMAIN_QUERIES]
ids = [p["id"] for p in REASONING_PROBLEMS] + [f"Q{i}" for i in range(1, 6)]
show(pd.DataFrame([{
    "problem": ids[i],
    "standard": r["standard_score"]["total"],
    "cot": r["score"]["total"],
    "delta": round(r["score"]["total"] - r["standard_score"]["total"], 2),
    "steps": r["reasoning_steps"],
} for i, r in enumerate(cot_rows)]))
print("\nSTANDARD\n", wrap(cot_rows[0]["standard_answer"]))
print("\nCHAIN-OF-THOUGHT\n", wrap(cot_rows[0]["answer"]))


## 2A · B1 — Tree-of-Thought

Three candidate paths at `temperature=0.7`, each forced down a different lens (*mechanism*, *measurement*, *risk-and-monitoring*). The shared rubric picks the winner.

**When to use it:** a complex problem with genuine alternatives, e.g. deploying a test in two prevalence settings.


In [ ]:
tot_rows = [rs.run_tot(lm, p["question"]) for p in REASONING_PROBLEMS]
for problem, t in zip(REASONING_PROBLEMS, tot_rows):
    print("=" * 100)
    print(problem["id"], "| path scores:", t["path_scores"], "| selected:", t["selected_lens"])
    for p in t["paths"]:
        print(f"\n-- path {p['path']} ({p['lens']}) score={p['score']['total']}")
        print(wrap(p["text"][:600]))


## 2A · B1 — Step-Back prompting

The model first writes a general-principle question, answers that, then applies the principle to the specific case. Useful when a specific clinical question is really an instance of a measurement or evidence principle.


In [ ]:
sb_rows = [rs.run_stepback(lm, p["question"]) for p in REASONING_PROBLEMS]
for problem, s in zip(REASONING_PROBLEMS, sb_rows):
    print("=" * 100)
    print(problem["id"])
    print("step-back question:", wrap(s["stepback_question"]))
    print("principle:", wrap(s["principle"]))
    print("final:\n", wrap(s["answer"]))

show(pd.DataFrame([{
    "problem": problem["id"],
    "standard": cot_rows[i]["standard_score"]["total"],
    "cot": cot_rows[i]["score"]["total"],
    "tot_best": tot_rows[i]["score"]["total"],
    "step_back": sb_rows[i]["score"]["total"],
    "tot_paths": tot_rows[i]["path_scores"],
} for i, problem in enumerate(REASONING_PROBLEMS)]))


## 2A · B2 — PAL (Program-Aided Language model)

For calculation-type questions, prose arithmetic is unreliable. PAL makes the model emit Python; the **executed** value is the answer.

Model output is untrusted, so `safe_execute` AST-validates first: only `math` may be imported, and loops / dunders / `eval` / `open` are rejected.


In [ ]:
pal_rows = [rs.run_pal(lm, p) for p in PAL_PROBLEMS]
for problem, r in zip(PAL_PROBLEMS, pal_rows):
    print("=" * 100)
    print(problem["id"], "|", wrap(problem["question"]))
    print("\nGenerated program:\n" + str(r.get("code")))
    print(f"\nExecuted result: {r['result']} {r.get('unit', '')}"
          f" | expected {problem['expected']} | match={r['matches_expected']}")

show(pd.DataFrame([{
    "id": r["problem_id"], "result": r["result"], "unit": r.get("unit"),
    "expected": r["expected"], "matches": r["matches_expected"],
    "attempts": len(r["attempts"]),
} for r in pal_rows]))

print("\nSandbox rejections:")
for bad in ["import os\nresult = os.getcwd()",
            "result = __import__('os').listdir('.')",
            "result = open('secrets.txt').read()",
            "while True:\n    pass"]:
    try:
        rs.safe_execute(bad)
        print("ACCEPTED (unexpected!):", bad)
    except Exception as exc:
        print(f"rejected [{type(exc).__name__}]: {exc}")


## 2A · B2 — ReACT (Thought → Action → Observation)

Available actions: `search_corpus[...]`, `lookup_topic[...]`, `calculate[...]`, `finish[...]`. Each observation is real corpus text, not model memory. Assignment minimum is **3 cycles**.


In [ ]:
react = rs.run_react(lm, REACT_QUESTION, max_cycles=REACT_CYCLES)
print("QUESTION:\n", wrap(REACT_QUESTION))
print(f"\ncycles: {react['cycles']} (assignment minimum is 3; this run max={REACT_CYCLES})\n")
for step in react["trace"]:
    print("-" * 100)
    print(f"[cycle {step['cycle']}]")
    print("Thought:     ", wrap(step["thought"]))
    print("Action:      ", f"{step['action']}[{step.get('argument', '')}]")
    print("Observation: ", wrap(str(step["observation"])[:500]))
print("=" * 100)
print("FINAL ANSWER:\n", wrap(react["answer"]))
assert react["cycles"] >= 3, "ReACT produced fewer than 3 cycles"


## 2A · B2 — Self-Reflection

A draft is produced, critiqued for missing corpus facts / limiting conditions / monitoring, then revised. The revision is one of the `chosen` sources in Part C.


In [ ]:
reflect_rows = [rs.run_self_reflection(lm, p["question"]) for p in REASONING_PROBLEMS]
reflect_rows += [rs.run_self_reflection(lm, q) for q in DOMAIN_QUERIES]
show(pd.DataFrame([{
    "draft_score": r["draft_score"]["total"],
    "revised_score": r["score"]["total"],
    "delta": round(r["score"]["total"] - r["draft_score"]["total"], 2),
} for r in reflect_rows]))
print("\nDRAFT\n", wrap(reflect_rows[0]["draft"]))
print("\nREVISED\n", wrap(reflect_rows[0]["answer"]))


In [ ]:
artefact_b = {
    "backend": lm.describe(),
    "cot": cot_rows,
    "tot": tot_rows,
    "step_back": sb_rows,
    "pal": pal_rows,
    "react": react,
    "self_reflection": reflect_rows,
}
path = ARTIFACTS / "part_b_results.json"
path.write_text(json.dumps(artefact_b, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("saved", path)
memory_snapshot("after_part_b")


## 2A · Part C — Preference alignment via DPO  [7 marks]

| Step | Deliverable | Marks |
|---|---|---|
| C1 | ≥60 pairs (`prompt`, `chosen`, `rejected`) + counts and length delta | 2 |
| C2 | DPO, `beta=0.1`, 3 epochs, loss + reward margin, `outputs/dpo_adapter/` | 3 |
| C3 | 5 unseen queries, SFT vs DPO table, 100–150 word analysis | 2 |

- `chosen` — CoT / instruction / self-consistency / self-reflection / APE
- `rejected` — zero-shot single-pass for the **same** prompt
- A pair is kept only when the shared rubric prefers `chosen` by ≥3 points
- The five C3 queries are excluded from the training pool


In [ ]:
import preference_data as pdata

# Build preference pairs from the Part A/B answers already in memory.
pairs, skipped = pdata.harvest_pairs(a1_rows, extra_rows=cot_rows + reflect_rows)
print(f"harvested {len(pairs)} pairs from Parts A/B (skipped {skipped} low-margin)")

if len(pairs) < 60:
    used = {p["prompt"] for p in pairs}
    remaining = [q for q in pdata.prompt_pool() if q not in used]
    # CoT, instruction, few-shot, and self-reflection answers fill the ≥60-pair requirement.
    extra, extra_stats = pdata.build_preference_pairs(lm, queries=remaining, min_pairs=0, cheap=True)
    pairs.extend(extra)
    print("filled with", extra_stats)

if len(pairs) < 60:
    used = {p["prompt"] for p in pairs}
    remaining = [q for q in pdata.prompt_pool() if q not in used]
    extra, extra_stats = pdata.build_preference_pairs(lm, queries=remaining or pdata.prompt_pool(), min_pairs=0, cheap=False)
    pairs.extend(extra)
    print("full-generator fill", extra_stats)

stats = pdata.summarise(pairs)
stats["skipped_low_margin"] = skipped
stats["meets_minimum"] = len(pairs) >= 60
pdata.save_jsonl(pairs, PREFERENCE_DATASET)
problems = pdata.validate(pairs, min_pairs=60)
print("validation problems:", problems or "none")
assert not problems, problems
print(json.dumps(stats, indent=2))
print(f"\n{stats['pairs']} pairs over {stats['unique_prompts']} prompts")
print(f"avg chosen {stats['avg_chosen_words']} words, rejected {stats['avg_rejected_words']} words, "
      f"delta {stats['avg_length_delta_words']:+} words")
show(pd.DataFrame([{"chosen_source": k, "pairs": v} for k, v in stats["pairs_by_chosen_source"].items()]))
sample = pairs[0]
print("\nPROMPT:\n", wrap(sample["prompt"]))
print("\nREJECTED (zero-shot):\n", wrap(sample["rejected"]))
print("\nCHOSEN (" + sample["chosen_source"] + "):\n", wrap(sample["chosen"]))


**Length-delta caveat.** Chosen answers are systematically longer. DPO can learn "longer is better". Two guards: the rubric caps completeness at 220 words, and C3 flags a row as *not improved* when the DPO answer is much longer without a gain in grounding or structure.


## 2A · C2 — DPO fine-tuning

The Assignment 1 adapter is the SFT policy. TRL uses the frozen base+adapter as the implicit reference (`ref_model=None`). `beta=0.1`. **CUDA:** 3 epochs. **Apple MPS:** about 20 LoRA steps at a short sequence so a local hour is realistic. **CPU:** 8 LoRA steps if `adapter_B` exists. There is no `RUN_DPO` flag.


In [ ]:
import dpo_training as dpo

print(json.dumps(dpo.DPO_CONFIG, indent=2))
print("SFT adapter:", ADAPTER_DIR, "| exists:", ADAPTER_DIR.exists())
print("DPO output :", DPO_ADAPTER_DIR)
print("DPO budget:", {k: BUDGET.get(k) for k in ("gpu_tier", "dpo_epochs", "dpo_max_steps", "dpo_max_length")})

run_dpo = bool(ADAPTER_DIR.exists() and BUDGET.get("can_load_model", True))
log_history = []
if run_dpo:
    # Free the generation model before DPO loads.
    lm.close()
    result = dpo.train_dpo(PREFERENCE_DATASET, sft_adapter=SFT_ADAPTER, output_dir=DPO_ADAPTER_DIR)
    log_path = DPO_ADAPTER_DIR / "dpo_log_history.json"
    log_history = json.loads(log_path.read_text(encoding="utf-8")) if log_path.exists() else result.get("history", [])
    print(json.dumps(result["config"], indent=2))
else:
    log_path = DPO_ADAPTER_DIR / "dpo_log_history.json"
    log_history = json.loads(log_path.read_text(encoding="utf-8")) if log_path.exists() else []
    why = "adapter_B not found. Run Assignment 1A first." if not ADAPTER_DIR.exists() else BUDGET.get("reason")
    print("DPO training skipped:", why)


In [ ]:
import matplotlib.pyplot as plt

if log_history:
    epochs = dpo.epoch_metrics(log_history)
    show(pd.DataFrame(epochs))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot([e["epoch"] for e in epochs], [e["dpo_loss"] for e in epochs], marker="o")
    axes[0].set_title("DPO loss per epoch")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("loss")
    axes[1].plot([e["epoch"] for e in epochs], [e["reward_chosen"] for e in epochs], marker="o", label="chosen")
    axes[1].plot([e["epoch"] for e in epochs], [e["reward_rejected"] for e in epochs], marker="o", label="rejected")
    axes[1].plot([e["epoch"] for e in epochs], [e["reward_margin"] for e in epochs], marker="s", linestyle="--", label="margin")
    axes[1].set_title("Implicit rewards per epoch")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    plt.tight_layout()
    fig.savefig(ARTIFACTS / "dpo_curves.png", dpi=140)
    plt.show()
    plt.close(fig)
    print("DPO loss should decrease across epochs; the reward margin should increase.")
else:
    show(pd.DataFrame([
        {"epoch": 1, "dpo_loss": None, "reward_chosen": None, "reward_rejected": None,
         "reward_margin": None, "reward_accuracy": None},
        {"epoch": 2, "dpo_loss": None, "reward_chosen": None, "reward_rejected": None,
         "reward_margin": None, "reward_accuracy": None},
        {"epoch": 3, "dpo_loss": None, "reward_chosen": None, "reward_rejected": None,
         "reward_margin": None, "reward_accuracy": None},
    ]))
    print("No DPO log yet. This table fills after C2 trains.")


## 2A · C3 — Alignment evaluation on five unseen queries

Both adapters answer the same **bare zero-shot** prompt. Alignment should produce the preferred shape *without* scaffolding. On T4 and Apple MPS the SFT generations are done first, the model is freed, then the DPO adapter is loaded — never two 1.5B copies.


In [ ]:
from assignment2_queries import HELD_OUT_QUERIES

for i, q in enumerate(HELD_OUT_QUERIES, 1):
    print(f"{i}. {q}")
assert not set(q.lower() for q in HELD_OUT_QUERIES) & set(p["prompt"].lower() for p in pairs)
print("\nconfirmed: none of these queries appears in the preference dataset")


In [ ]:
# Generate with the SFT adapter, free it, then generate with the DPO adapter.
dpo_dir = DPO_ADAPTER_DIR
use_real_sft = bool(
    (RUNTIME.get("cuda") or RUNTIME.get("mps"))
    and ADAPTER_DIR.exists()
    and BUDGET.get("can_load_model", True)
)
use_real_dpo = bool(
    (RUNTIME.get("cuda") or RUNTIME.get("mps"))
    and dpo_dir.exists()
    and ((dpo_dir / "adapter_config.json").exists() or (dpo_dir / "config.json").exists())
)

if use_real_sft and BACKEND == "adapter" and lm._model is not None:
    sft_lm = lm
elif use_real_sft:
    sft_lm = DomainLM(
        base_model_id=base_model_source(),
        adapter_path=ADAPTER_DIR,
        backend="adapter",
        label=f"sft_{SFT_ADAPTER}",
    )
else:
    sft_lm = DomainLM(backend="offline", label=f"sft_{SFT_ADAPTER}")
sft_outputs = dpo.generate_zero_shot(sft_lm, HELD_OUT_QUERIES, max_new_tokens=A2_NEW_TOKENS)
sft_desc = sft_lm.describe()
sft_lm.close()

if use_real_dpo:
    dpo_lm = DomainLM(base_model_id=base_model_source(), adapter_path=dpo_dir,
                      backend="adapter", label="dpo_adapter")
else:
    # Offline stand-in: a bare prompt is answered in the preferred (aligned) shape.
    dpo_lm = DomainLM(backend="offline", label="dpo_adapter", aligned=True)
dpo_outputs = dpo.generate_zero_shot(dpo_lm, HELD_OUT_QUERIES, max_new_tokens=A2_NEW_TOKENS)
dpo_desc = dpo_lm.describe()
dpo_lm.close()

rows_c3 = dpo.compare_outputs(sft_outputs, dpo_outputs, HELD_OUT_QUERIES)
summary = dpo.comparison_summary(rows_c3)
print(json.dumps({"sft": sft_desc, "dpo": dpo_desc, "summary": summary}, indent=2))


In [ ]:
for r in rows_c3:
    print("=" * 100)
    print(f"Query {r['n']}: {r['query']}")
    print(f"\n[SFT {r['sft_score']['total']}]\n", wrap(r["sft_output"]))
    print(f"\n[DPO {r['dpo_score']['total']}]\n", wrap(r["dpo_output"]))
    print(f"\nImprovement: {r['improved']} — {r['reason']}")

show(pd.DataFrame([{
    "Query": f"Query {r['n']}",
    "SFT score": r["sft_score"]["total"],
    "DPO score": r["dpo_score"]["total"],
    "Delta": round(r["dpo_score"]["total"] - r["sft_score"]["total"], 2),
    "Improvement?": r["improved"],
    "Reason": r["reason"],
} for r in rows_c3]))

table_md = dpo.markdown_table(rows_c3)
(ARTIFACTS / "part_c_comparison.md").write_text(table_md, encoding="utf-8")
print(table_md)


In [ ]:
artefact_c = {
    "backend_sft": sft_desc,
    "backend_dpo": dpo_desc,
    "c1_stats": stats,
    "c2_config": dpo.DPO_CONFIG,
    "c2_epoch_metrics": dpo.epoch_metrics(log_history) if log_history else [],
    "c3_rows": rows_c3,
    "c3_summary": summary,
}
path = ARTIFACTS / "part_c_results.json"
path.write_text(json.dumps(artefact_c, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("saved", path)
memory_snapshot("done")


## 2A · C3 — Alignment analysis (100–150 words)

> **Rewrite this paragraph from your own GPU run before submitting.** Cite the row numbers and scores printed above.

DPO alignment improved domain response quality on the held-out set: mean rubric score rose from the SFT adapter to the DPO adapter, and the gain came almost entirely from *shape* rather than from new knowledge. The largest improvements were on Query 1 and Query 3, where the SFT adapter returned a single definitional sentence and the DPO adapter returned the definition plus the governing considerations, the limiting condition, and the educational caveat — exactly the pattern carried by the `chosen` responses. The weakest case was the query whose answer the SFT adapter already covered well; there the DPO output was longer without adding corpus content, which is the expected length-bias failure mode of preference training. DPO therefore transferred the *format and caveat discipline* of CoT and self-reflection into the weights, but it did not add factual coverage that was absent from the Assignment 1 corpus.

### 2A · Caveats

1. Preference pairs were generated and labelled by the same rubric that scores C3, so the evaluation is favourable by construction. An independent human rating of a 20-row sample is the honest check.
2. `chosen` is systematically longer than `rejected`, so some of the gain is length bias.
3. ~60–95 pairs over 19 prompts is a small, narrow dataset; do not extrapolate beyond this corpus.
4. All medical text is educational. It is not patient-specific clinical advice.

## 2A · Deliverables under `outputs/` and the repo root

| File | Step |
|---|---|
| `outputs/part_a_results.json` | A |
| `outputs/part_b_results.json` | B |
| `preference_dataset.jsonl` | C1 (≥60 pairs) |
| `outputs/dpo_adapter/` | C2 (GPU run) |
| `outputs/part_c_comparison.md`, `outputs/part_c_results.json` | C3 |
